<a href="https://colab.research.google.com/github/CSI5195-Project/Fraud-Detection/blob/main/SNN_Fraud_Detection_Model1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

danieldumi_bank_account_fraud_dataset_path = kagglehub.dataset_download('danieldumi/bank-account-fraud-dataset')
danieldumi_hyperparameters_path = kagglehub.utility_script_install('danieldumi/hyperparameters')
danieldumi_utils_path = kagglehub.utility_script_install('danieldumi/utils')
danieldumi_modelsnnpc_path = kagglehub.utility_script_install('danieldumi/modelsnnpc')
danieldumi_snn_metrics_path = kagglehub.utility_script_install('danieldumi/snn-metrics')
evavivante_snn_metrics_2_path = kagglehub.utility_script_install('evavivante/snn-metrics-2')
evavivante_modelsnnpc_withproba_path = kagglehub.utility_script_install('evavivante/modelsnnpc-withproba')

print('Data source import complete.')


In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
from datetime import datetime

import hyperparameters
from utils import RandomTrial, read_data

In [ ]:
from modelsnnpc_withproba import *

In [ ]:
DATASET_LIST = ["Variant I"]
# DATASET_LIST = ["Base", "Variant I", "Variant II", "Variant III", "Variant IV", "Variant V"]
NUM_TRIALS = 1
BEGIN_TRIAL = 0
BASE_SEED = 42

METRICS_NAME_GLOBAL = ["accuracy", "precision", "recall", "fpr", "f1_score","auc"]
METRICS_NAME_5FPR = ["accuracy@5FPR","precision@5FPR", "recall@5FPR", "fpr@5FPR", "f1_score@5FPR"]
METRICS_FAIRNESS = ["fpr_ratio_age", "fpr_ratio_income", "fpr_ratio_employment",
                   "eod_age", "eod_income", "eod_employment",
                   "aod_age", "aod_income", "aod_employment"]
HYPERPARAMETERS = hyperparameters.P20_S50

EXPERIMENT_NAME = f"P{HYPERPARAMETERS['population']}-S{HYPERPARAMETERS['step']}-{NUM_TRIALS}trials-begin{BEGIN_TRIAL}"
FIXED_DATE = None

In [ ]:
def dataset_loop(train_dfs, test_dfs, dataset_name, trial_number, seed, path, runs):
    train_dfs[dataset_name].to_csv(f'train_dfs{trial_number}.csv', index=False)
    test_dfs[dataset_name].to_csv(f'test_dfs{trial_number}.csv', index=False)
    x_train = train_dfs[dataset_name].drop(columns=["fraud_bool"])
    y_train = train_dfs[dataset_name]["fraud_bool"]
    x_test = test_dfs[dataset_name].drop(columns=["fraud_bool"])
    y_test = test_dfs[dataset_name]["fraud_bool"]
    num_classes = len(np.unique(y_train))
    num_features = len(x_train.columns)
    class_weights = (1-HYPERPARAMETERS['weight'], HYPERPARAMETERS['weight'])
    model = ModelSNNPC(
        num_features=num_features,
        num_classes=num_classes,
        class_weights=class_weights,
        betas=HYPERPARAMETERS['beta'],
        slope=HYPERPARAMETERS['slope'],
        thresholds=HYPERPARAMETERS['threshold'],
        population=HYPERPARAMETERS['population'],
        batch_size=HYPERPARAMETERS['batch'],
        num_epochs=HYPERPARAMETERS['epoch'],
        num_steps=HYPERPARAMETERS['step'],
        adam_betas=HYPERPARAMETERS['adam_beta'],
        learning_rate=HYPERPARAMETERS['learning_rate'],
        verbose=0
    )
    model.fit(x_train, y_train)
    predictions, targets, proba = model.predict(x_test, y_test)
    np.savetxt(f"predictions_run{trial_number}.csv", predictions, delimiter=",", fmt="%d")
    np.savetxt(f"targets_run{trial_number}.csv", targets, delimiter=",", fmt="%d")
    np.savetxt(f"proba_run{trial_number}.csv", proba, delimiter="\t")
    metrics = model.evaluate(targets, predictions)
    metrics_aequitas = model.evaluate_business_constraint(targets, predictions)
    metrics.update(metrics_aequitas)
    fairness_age = model.evaluate_fairness(x_test, targets, predictions, "customer_age", 50)
    metrics.update({k+"_age": v for k, v in fairness_age.items()})
    fairness_income = model.evaluate_fairness(x_test, targets, predictions, "income", 0.5)
    metrics.update({k+"_income": v for k, v in fairness_income.items()})
    fairness_employement = model.evaluate_fairness(x_test, targets, predictions, "employment_status", 3)
    metrics.update({k+"_employment": v for k, v in fairness_employement.items()})
    results = {}
    results["dataset"] = dataset_name
    results["trial"] = trial_number
    results["seed"] = seed
    for metric in METRICS_NAME_GLOBAL:
        results[metric] = metrics[metric]
    for metric in METRICS_NAME_5FPR:
        results[metric] = metrics_aequitas[metric]
    for metric in METRICS_FAIRNESS:
        results[metric] = metrics[metric]
    csv_row = ','.join([str(x) for x in results.values()])
    with open(path, "a") as f:
        f.write(f"{csv_row}\n")
    prev_runs = runs.get(dataset_name, [])
    prev_runs.append(results)
    print(results)
    runs[dataset_name] = prev_runs
    return runs


In [ ]:
def simulation(datasets, train_dfs, test_dfs, path="./results.csv"):
    np.random.seed(BASE_SEED)
    seeds = np.random.choice(list(range(1_000_000)), size=NUM_TRIALS, replace=False)
    runs = {}
    for trial in range(NUM_TRIALS):
        seed = seeds[trial]
        trial_number = trial
        trial = RandomTrial(seed=seed)
        if trial_number < BEGIN_TRIAL:
            print(f"Skipping trial {trial_number} – seed {seed}")
            continue
        for dataset_name in datasets.keys():
            print(f"Running trial {trial_number} with seed {seed} on dataset {dataset_name}")
            runs = dataset_loop(train_dfs, test_dfs, dataset_name, trial_number, seed, path, runs)
    return runs

In [ ]:
base_path = "/kaggle/input/bank-account-fraud-dataset/"
_, datasets, train_dfs, test_dfs = read_data(base_path, DATASET_LIST)
if not FIXED_DATE:
    date = datetime.now().strftime("%Y%m%d_%H%M%S")
else:
    date = FIXED_DATE
experiment_dir = f"/kaggle/working/results/{date}-{EXPERIMENT_NAME}"
results_path = f"{experiment_dir}/results.csv"
os.makedirs(experiment_dir, exist_ok=True)
if not os.path.exists(results_path):
    with open(results_path, "w") as f:
        f.write("dataset,trial,seed,accuracy,precision,recall,fpr,f1_score,auc,accuracy@5FPR,precision@5FPR,recall@5FPR,fpr@5FPR,f1_score@5FPR,fpr_ratio_age,fpr_ratio_income,fpr_ratio_employment,eod_age,eod_income,eod_employment,aod_age,aod_income,aod_employment\n")

In [ ]:
simulation(datasets, train_dfs, test_dfs, path=results_path)